# Tire Analysis: Temperature Distribution & Thermography

This notebook visualizes tire temperature data from infrared sensors across your tires, helping you understand tire behavior and identify setup issues.

## What You'll Find Here

- **Tire Temperature Heatmaps**: Visual representation of temperature distribution across all four tires (FL, FR, RL, RR) throughout the lap
- **Temperature vs Distance**: See how tire temperatures change through different track sections
- **Speed & G-Force Overlay**: Correlate tire temperatures with speed and combined lateral/longitudinal acceleration
- **Driver Inputs Overlay**: See throttle, brake, and steering inputs aligned with temperature data

## How to Interpret the Results

### Temperature Heatmaps
- **Sensor zones**: Represent temperature sensor positions across the tire width
  - FL/RL: Zone 1 = Outside (left), last zone = Inside (right)
  - FR/RR: Zone 1 = Inside (left), last zone = Outside (right)
- **Hot spots (bright)**: Areas of high temperature - could indicate excessive load or slip
- **Cold spots (dark)**: Areas not being worked - potential grip left on the table
- **Even temperature gradient**: Indicates good tire usage and camber settings

### Setup Insights
- **Outside edge hot**: May need more negative camber
- **Inside edge hot**: May have too much negative camber
- **Center hot**: Could indicate over-inflation
- **Edges hot, center cold**: Could indicate under-inflation
- **Front vs Rear difference**: Balance insights for understeer/oversteer tendencies

## Using Your Own Data

To analyze your own data:

1. **Run the first cell** below to install packages and display the upload widget
2. **Click "Choose File"** to select your `.xrk`, `.xrz`, or `.ibt` file
3. **Run all remaining cells** to analyze your data

The status indicator will show which file is being used. If you don't upload a file, the sample data will be used.

## Requirements

- Tire temperature channels (configured via the tire channel picker — e.g. `FL_Ch1`-`FL_Ch8` for AIM, or `LFtempL`/`LFtempM`/`LFtempR` for iRacing)
- GPS Speed channel for distance calculation
- Acceleration data (configured via channel picker)
- Driver input channels (configured via channel picker)

**Note:** This notebook works in both JupyterLite (browser) and standard JupyterLab environments.

In [ ]:
# Install required packages (needed for JupyterLite, skipped in regular JupyterLab if already installed)
%pip install -q motorsports-data-notebook

# Use the Rust parser backend for ~3x faster file loading
import os

os.environ["LIBXRK_BACKEND"] = "rust"

# Import helper functions
from motorsports_data_notebook.visualization import (
    format_lap_time,
    plot_tire_thermography,
    show_fig,
)
from motorsports_data_notebook.widgets import SessionPicker

# Session picker with channel and tire temperature configuration
# Upload your own file and select a lap to analyze
# Tire temperature channels are auto-configured from vehicle profiles
session = SessionPicker(
    default_file="../data/CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz",
    channel_mapping={
        "lateral_g": "LateralAcc",
        "inline_g": "InlineAcc",
        "throttle": "PPS",
        "brake": "BrakePress",
        "steering": "SteerAngle",
    },
    tire_channel_mapping={
        "FL": [f"FL_Ch{i}" for i in range(1, 9)],
        "FR": [f"FR_Ch{i}" for i in range(1, 9)],
        "RL": [f"RL_Ch{i}" for i in range(1, 9)],
        "RR": [f"RR_Ch{i}" for i in range(1, 9)],
    },
)
session.display()

In [ ]:
# Get laps as pandas DataFrame for display
laps = session.get_laps()

In [ ]:
# Display lap times table
laps.style.format({"lap_time": format_lap_time})  # type: ignore[dict-item]

In [ ]:
# Get the configured channel names and session data
log = session.get_log()
CHANNEL_NAMES = session.get_channel_names()

# Collect tire temperature channels — only include channels that exist in the log
available = set(log.channels.keys())
tire_channels = [
    v for k, v in CHANNEL_NAMES.items() if k.startswith("tire_temp_") and v in available
]

other_channels = [
    "distance_m",
    "speed_kmh",
    CHANNEL_NAMES["lateral_g"],
    CHANNEL_NAMES["inline_g"],
    CHANNEL_NAMES["brake"],
    CHANNEL_NAMES["throttle"],
    CHANNEL_NAMES["steering"],
]

# Extract data for the selected lap using libxrk 0.5.0 methods
selected_lap = session.get_selected_lap()
lap_num = int(selected_lap["num"])

# Filter to lap, select channels, and resample to distance_m timebase
channels = (
    log.filter_by_lap(lap_num)
    .select_channels(tire_channels + other_channels)
    .resample_to_channel("distance_m")
    .channels
)

In [ ]:
# Tire Thermography - Selected Lap
fig = plot_tire_thermography(
    channels, CHANNEL_NAMES, title=f"Tire Temperatures - Lap {int(selected_lap['num'])}"
)
show_fig(fig)